# Keyframe pipeline — Kaggle T4x2

Uses the project code and PE-Core checkpoint already attached as Kaggle inputs (read-only), copies
the code into `/kaggle/working` (writable, needed for `.venv`/outputs), then runs
`scripts/run_kaggle_t4x2.sh` once per dataset URL — shards each dataset across both T4s
(2 independent processes, no DDP), reusing the local PE-Core checkpoint so nothing downloads.

In [ ]:
import torch
assert torch.cuda.device_count() == 2, f"expected 2 GPUs, got {torch.cuda.device_count()} -- enable accelerator 'GPU T4 x2' in notebook settings and restart"
!nvidia-smi --query-gpu=index,name,memory.total --format=csv,noheader

In [ ]:
CODE_SRC = "/kaggle/input/datasets/meowluvmatcha/keyframe-pipeline-global-9"
MODEL_DIR = "/kaggle/input/models/meowluvmatcha/lufina/transformers/default/1/pecore_vision_only"

import os
assert os.path.isdir(CODE_SRC), f"code input not found: {CODE_SRC}"
assert os.path.isfile(f"{MODEL_DIR}/model.safetensors"), f"model.safetensors not found under {MODEL_DIR}"

!rm -rf /kaggle/working/keyframe_pipeline
!cp -r "{CODE_SRC}" /kaggle/working/keyframe_pipeline
%cd /kaggle/working/keyframe_pipeline
!bash setup.sh

In [ ]:
URLS = [
    "https://aic-data.ledo.io.vn/Videos_L26_d.zip",
    "https://aic-data.ledo.io.vn/Videos_L26_e.zip",
    "https://aic-data.ledo.io.vn/Videos_L28_a.zip",
]

for url in URLS:
    print(f"\n=== {url} ===")
    !bash scripts/run_kaggle_t4x2.sh --url "{url}" --model-source local --model-dir "{MODEL_DIR}"

In [ ]:
!ls -la /kaggle/working/*_results.zip
!echo '--- embedding files per dataset ---'
!for d in /kaggle/working/output_*; do echo "$d: $(find "$d" -name embeddings.npy | wc -l)"; done